# Lesson 2: Traditional NER with spaCy

## 🎯 Learning Objectives

By the end of this lesson, you will:
1. Master spaCy's pre-trained NER models
2. Understand spaCy's NER architecture and pipeline
3. Create custom entity rulers for rule-based NER
4. Visualize entities with displaCy
5. Train a custom NER model from scratch
6. Combine rule-based and statistical approaches

---

## 📚 Table of Contents

1. [Introduction to spaCy](#1-introduction-to-spacy)
2. [Using Pre-trained NER Models](#2-using-pre-trained-ner-models)
3. [Understanding the NER Pipeline](#3-understanding-the-ner-pipeline)
4. [Entity Rulers: Rule-Based NER](#4-entity-rulers-rule-based-ner)
5. [Visualizing Entities with displaCy](#5-visualizing-entities-with-displacy)
6. [Training Custom NER Models](#6-training-custom-ner-models)
7. [Best Practices & Tips](#7-best-practices--tips)
8. [Further Reading](#8-further-reading)

---

## 📦 Setup & Installation

In [ ]:
# Install required packages
!pip install -q spacy==3.7.2
!python -m spacy download en_core_web_sm -q
!python -m spacy download en_core_web_md -q
!python -m spacy download en_core_web_lg -q

In [ ]:
# Import libraries
import spacy
from spacy import displacy
from spacy.tokens import Span
from spacy.language import Language
from spacy.pipeline import EntityRuler
from spacy.training import Example
from spacy.util import minibatch, compounding
import random
import warnings
warnings.filterwarnings('ignore')

print(f"✅ spaCy version: {spacy.__version__}")

---

## 1. Introduction to spaCy

### What is spaCy?

> **spaCy** is an open-source library for advanced Natural Language Processing in Python. It's designed specifically for production use and is known for its speed and accuracy.
>
> — [spaCy Official Documentation](https://spacy.io/)

### Why spaCy for NER?

| Feature | Benefit |
|---------|--------|
| **Speed** | Fastest NLP library, optimized Cython code |
| **Accuracy** | State-of-the-art statistical models |
| **Pre-trained Models** | Ready-to-use models for 20+ languages |
| **Easy Integration** | Works well with deep learning frameworks |
| **Production Ready** | Designed for real-world applications |
| **Active Development** | Regular updates and improvements |

### spaCy Model Sizes

| Model | Size | Vectors | Accuracy |
|-------|------|---------|----------|
| `en_core_web_sm` | 12 MB | No word vectors | Good |
| `en_core_web_md` | 40 MB | 20k word vectors | Better |
| `en_core_web_lg` | 560 MB | 685k word vectors | Best |
| `en_core_web_trf` | 438 MB | Transformer-based | State-of-the-art |

> **Reference**: [spaCy Models Documentation](https://spacy.io/models/en)

In [ ]:
# Load models and compare
nlp_sm = spacy.load("en_core_web_sm")
nlp_md = spacy.load("en_core_web_md")
nlp_lg = spacy.load("en_core_web_lg")

print("📊 Model Comparison\n")
print("=" * 60)

for name, nlp in [("sm", nlp_sm), ("md", nlp_md), ("lg", nlp_lg)]:
    # Get pipeline components
    components = list(nlp.pipe_names)
    # Get NER labels
    ner_labels = nlp.get_pipe("ner").labels if "ner" in nlp.pipe_names else []
    
    print(f"\nen_core_web_{name}:")
    print(f"   Pipeline: {components}")
    print(f"   NER Labels: {len(ner_labels)} types")

---

## 2. Using Pre-trained NER Models

### Basic Usage

In [ ]:
# Load the medium model for good balance of speed and accuracy
nlp = spacy.load("en_core_web_md")

# Process text
text = """
Apple Inc. announced that CEO Tim Cook will present the new iPhone 15 
at their headquarters in Cupertino, California on September 12, 2023. 
The event is expected to generate $50 billion in revenue.
"""

doc = nlp(text)

# Extract entities
print("🏷️ Extracted Entities:\n")
print(f"{'Entity':<30} {'Label':<12} {'Description'}")
print("=" * 70)

for ent in doc.ents:
    print(f"{ent.text:<30} {ent.label_:<12} {spacy.explain(ent.label_)}")

In [ ]:
# Accessing entity attributes
print("📋 Detailed Entity Information:\n")

for ent in doc.ents:
    print(f"Text: '{ent.text}'")
    print(f"   Label: {ent.label_} ({spacy.explain(ent.label_)})")
    print(f"   Start char: {ent.start_char}, End char: {ent.end_char}")
    print(f"   Start token: {ent.start}, End token: {ent.end}")
    print(f"   Root token: {ent.root.text}")
    print()

In [ ]:
# Token-level entity information
print("🔤 Token-Level Entity Tags:\n")
print(f"{'Token':<15} {'ENT_TYPE':<12} {'ENT_IOB':<8} {'ENT_IOB_'}")
print("=" * 50)

for token in doc[:20]:  # First 20 tokens
    if token.ent_type_:  # Only show tokens that are part of entities
        print(f"{token.text:<15} {token.ent_type_:<12} {token.ent_iob:<8} {token.ent_iob_}")

### Understanding IOB Tags in spaCy

spaCy uses **BILUO** (also called BIOES) tagging internally:

| Tag | Meaning | Example |
|-----|---------|--------|
| `B` | Beginning of entity | `B-ORG` (first token of "Apple Inc.") |
| `I` | Inside entity | `I-ORG` (continuation token) |
| `L` | Last token of entity | `L-ORG` (last token of multi-word entity) |
| `U` | Unit (single token entity) | `U-GPE` ("California") |
| `O` | Outside any entity | Non-entity tokens |

In [ ]:
# Batch processing for efficiency
texts = [
    "Google was founded by Larry Page and Sergey Brin at Stanford University.",
    "Amazon's Jeff Bezos announced a $10 billion climate fund.",
    "Microsoft acquired LinkedIn for $26.2 billion in December 2016.",
    "Tesla CEO Elon Musk tweeted from Fremont, California."
]

print("⚡ Batch Processing Results:\n")

# Use nlp.pipe() for efficient batch processing
for doc in nlp.pipe(texts, batch_size=2):
    entities = [(ent.text, ent.label_) for ent in doc.ents]
    print(f"Text: {doc.text[:50]}...")
    print(f"Entities: {entities}\n")

---

## 3. Understanding the NER Pipeline

### spaCy Pipeline Architecture

```
Text → Tokenizer → [Component 1] → [Component 2] → ... → Doc
                          │              │
                        tagger          ner
                       parser         lemmatizer
```

The NER component is just one part of spaCy's processing pipeline.

In [ ]:
# Examine the pipeline
print("🔧 spaCy Pipeline Components:\n")

for i, (name, component) in enumerate(nlp.pipeline):
    print(f"{i+1}. {name}")
    print(f"   Type: {type(component).__name__}")
    
    # Get labels if available
    if hasattr(component, 'labels'):
        labels = component.labels
        print(f"   Labels: {len(labels)} types")
        if name == 'ner':
            print(f"   NER Labels: {list(labels)[:10]}...")
    print()

In [ ]:
# Disable unnecessary components for faster NER-only processing
import time

text = "Apple CEO Tim Cook announced new products in Cupertino." * 100

# Full pipeline
start = time.time()
doc_full = nlp(text)
time_full = time.time() - start

# NER only (disable other components)
start = time.time()
with nlp.select_pipes(enable=['ner']):
    doc_ner = nlp(text)
time_ner = time.time() - start

print("⏱️ Performance Comparison:\n")
print(f"Full pipeline:     {time_full:.4f}s")
print(f"NER only:          {time_ner:.4f}s")
print(f"Speedup:           {time_full/time_ner:.2f}x faster")

### How spaCy's NER Works

spaCy's NER uses a **transition-based** algorithm:

1. **Input**: Sequence of tokens with word vectors
2. **Neural Network**: Multi-layer perceptron (MLP) or CNN
3. **Transitions**: Predicts actions like:
   - `SHIFT`: Move to next token
   - `REDUCE`: Complete current entity
   - `OUT`: Token is not part of an entity
   - `ENTITY_TYPE`: Start/continue entity of given type

```
┌─────────────┐    ┌──────────────┐    ┌─────────────┐
│   Tokens    │ -> │  CNN/MLP     │ -> │ Transition  │
│  + Vectors  │    │  Encoder     │    │  Classifier │
└─────────────┘    └──────────────┘    └─────────────┘
                                              │
                                              ▼
                                       Entity Labels
```

> **Reference**: [spaCy EntityRecognizer Architecture](https://spacy.io/api/entityrecognizer)

---

## 4. Entity Rulers: Rule-Based NER

### Why Use Rules?

Sometimes you need to:
- Add entities the model doesn't know (product names, internal terms)
- Ensure specific patterns are always detected
- Handle domain-specific entities without retraining

### Creating an Entity Ruler

In [ ]:
# Create a blank model and add entity ruler
nlp_ruler = spacy.load("en_core_web_md")

# Create entity ruler
ruler = nlp_ruler.add_pipe("entity_ruler", before="ner")

# Define patterns
patterns = [
    # Simple exact match patterns
    {"label": "PRODUCT", "pattern": "iPhone 15"},
    {"label": "PRODUCT", "pattern": "MacBook Pro"},
    {"label": "PRODUCT", "pattern": "Apple Watch"},
    
    # Token-based patterns (more flexible)
    {"label": "PRODUCT", "pattern": [{"LOWER": "iphone"}, {"IS_DIGIT": True}]},
    {"label": "TECH_COMPANY", "pattern": [{"LOWER": {"IN": ["apple", "google", "microsoft", "amazon"]}}]},
    
    # Pattern with multiple tokens
    {"label": "FRAMEWORK", "pattern": [{"LOWER": "machine"}, {"LOWER": "learning"}]},
    {"label": "FRAMEWORK", "pattern": [{"LOWER": "deep"}, {"LOWER": "learning"}]},
]

ruler.add_patterns(patterns)

print("✅ Entity ruler created with custom patterns")

In [ ]:
# Test the entity ruler
test_texts = [
    "Apple announced the iPhone 15 at their machine learning conference.",
    "Google's deep learning team released new research.",
    "I bought a MacBook Pro and an Apple Watch yesterday."
]

print("🔍 Entity Ruler Results:\n")

for text in test_texts:
    doc = nlp_ruler(text)
    print(f"Text: {text}")
    print(f"Entities:")
    for ent in doc.ents:
        print(f"   • '{ent.text}' → {ent.label_}")
    print()

### Advanced Pattern Matching

spaCy's pattern matching supports rich token attributes:

In [ ]:
# Create a more sophisticated entity ruler
nlp_adv = spacy.load("en_core_web_sm")
ruler_adv = nlp_adv.add_pipe("entity_ruler", before="ner")

advanced_patterns = [
    # Match version numbers (e.g., "Python 3.10")
    {
        "label": "SOFTWARE",
        "pattern": [
            {"LOWER": {"IN": ["python", "java", "ruby", "node"]}},
            {"LIKE_NUM": True}
        ]
    },
    
    # Match email-like patterns
    {
        "label": "EMAIL",
        "pattern": [{"LIKE_EMAIL": True}]
    },
    
    # Match URLs
    {
        "label": "URL",
        "pattern": [{"LIKE_URL": True}]
    },
    
    # Match stock tickers (uppercase, 1-5 letters)
    {
        "label": "TICKER",
        "pattern": [{"TEXT": {"REGEX": "^[A-Z]{1,5}$"}, "IS_ALPHA": True}]
    },
    
    # Match titles followed by names
    {
        "label": "PERSON_TITLE",
        "pattern": [
            {"LOWER": {"IN": ["dr", "mr", "mrs", "ms", "prof"]}},
            {"TEXT": ".", "OP": "?"},
            {"IS_TITLE": True}
        ]
    }
]

ruler_adv.add_patterns(advanced_patterns)

# Test
test_text = "Dr. Smith recommends Python 3.10 for the project. Contact support@company.com or visit https://example.com. Watch AAPL and GOOGL stocks."
doc = nlp_adv(test_text)

print("🔍 Advanced Pattern Matching:\n")
print(f"Text: {test_text}\n")
for ent in doc.ents:
    print(f"   • '{ent.text}' → {ent.label_}")

### Pattern Syntax Reference

| Attribute | Description | Example |
|-----------|-------------|--------|
| `TEXT` | Exact text | `{"TEXT": "Hello"}` |
| `LOWER` | Lowercase text | `{"LOWER": "hello"}` |
| `IS_ALPHA` | Alphabetic | `{"IS_ALPHA": True}` |
| `IS_DIGIT` | Numeric | `{"IS_DIGIT": True}` |
| `IS_TITLE` | Titlecase | `{"IS_TITLE": True}` |
| `IS_UPPER` | Uppercase | `{"IS_UPPER": True}` |
| `LIKE_NUM` | Number-like | `{"LIKE_NUM": True}` |
| `LIKE_EMAIL` | Email-like | `{"LIKE_EMAIL": True}` |
| `LIKE_URL` | URL-like | `{"LIKE_URL": True}` |
| `POS` | Part-of-speech | `{"POS": "NOUN"}` |
| `LENGTH` | Token length | `{"LENGTH": {">=": 5}}` |
| `IN` | Match any in list | `{"LOWER": {"IN": ["a", "b"]}}` |
| `REGEX` | Regular expression | `{"TEXT": {"REGEX": "^[A-Z]+$"}}` |
| `OP` | Quantifier | `"?", "*", "+", "!"` |

> **Reference**: [spaCy Rule-based Matching](https://spacy.io/usage/rule-based-matching)

---

## 5. Visualizing Entities with displaCy

spaCy includes a powerful visualization tool called **displaCy**.

In [ ]:
# Basic entity visualization
nlp = spacy.load("en_core_web_md")

text = """
Elon Musk, the CEO of Tesla and SpaceX, announced on Twitter that 
the company would invest $1.5 billion in Bitcoin. The announcement 
was made from Austin, Texas on February 8, 2021.
"""

doc = nlp(text)

# Render entities (in Jupyter, this shows inline)
displacy.render(doc, style="ent", jupyter=True)

In [ ]:
# Customize colors and labels
colors = {
    "PERSON": "linear-gradient(90deg, #aa9cfc, #fc9ce7)",
    "ORG": "linear-gradient(90deg, #a8d5ba, #a8d5ba)",
    "GPE": "linear-gradient(90deg, #ffd89b, #ffd89b)",
    "MONEY": "linear-gradient(90deg, #84fab0, #8fd3f4)",
    "DATE": "linear-gradient(90deg, #a1c4fd, #c2e9fb)"
}

options = {
    "ents": ["PERSON", "ORG", "GPE", "MONEY", "DATE"],
    "colors": colors
}

displacy.render(doc, style="ent", jupyter=True, options=options)

In [ ]:
# Visualize multiple documents
texts = [
    "Amazon was founded by Jeff Bezos in Seattle in 1994.",
    "Google's headquarters are in Mountain View, California.",
    "Microsoft was founded by Bill Gates and Paul Allen."
]

docs = list(nlp.pipe(texts))

for doc in docs:
    displacy.render(doc, style="ent", jupyter=True)

In [ ]:
# Export as HTML
html = displacy.render(doc, style="ent", page=True)

# Save to file (uncomment to save)
# with open("entities.html", "w") as f:
#     f.write(html)

print(f"HTML length: {len(html)} characters")
print(f"\nFirst 500 characters of HTML:\n{html[:500]}...")

---

## 6. Training Custom NER Models

### When to Train Custom Models?

- Domain-specific entities (medical terms, legal jargon)
- New entity types not in pre-trained models
- Improve accuracy for your specific use case

### Training Data Format

spaCy expects training data in a specific format:

In [ ]:
# Training data format
TRAIN_DATA = [
    (
        "Python is a programming language created by Guido van Rossum.",
        {"entities": [(0, 6, "PROGRAMMING_LANG"), (45, 61, "PERSON")]}
    ),
    (
        "JavaScript was developed at Netscape by Brendan Eich.",
        {"entities": [(0, 10, "PROGRAMMING_LANG"), (28, 36, "ORG"), (40, 52, "PERSON")]}
    ),
    (
        "Java was created by James Gosling at Sun Microsystems.",
        {"entities": [(0, 4, "PROGRAMMING_LANG"), (20, 33, "PERSON"), (37, 53, "ORG")]}
    ),
    (
        "The Ruby programming language was designed by Yukihiro Matsumoto.",
        {"entities": [(4, 8, "PROGRAMMING_LANG"), (46, 64, "PERSON")]}
    ),
    (
        "Rust is developed by Mozilla and was created by Graydon Hoare.",
        {"entities": [(0, 4, "PROGRAMMING_LANG"), (21, 28, "ORG"), (48, 61, "PERSON")]}
    ),
    (
        "Go was designed at Google by Robert Griesemer, Rob Pike, and Ken Thompson.",
        {"entities": [(0, 2, "PROGRAMMING_LANG"), (19, 25, "ORG"), (29, 45, "PERSON"), (47, 55, "PERSON"), (61, 73, "PERSON")]}
    ),
    (
        "Swift is a programming language developed by Apple.",
        {"entities": [(0, 5, "PROGRAMMING_LANG"), (45, 50, "ORG")]}
    ),
    (
        "TypeScript was developed by Microsoft.",
        {"entities": [(0, 10, "PROGRAMMING_LANG"), (28, 37, "ORG")]}
    ),
]

print(f"📊 Training data: {len(TRAIN_DATA)} examples")
print(f"\nSample: {TRAIN_DATA[0]}")

In [ ]:
# Create a blank model and train NER
def train_custom_ner(train_data, n_iter=30):
    """Train a custom NER model"""
    
    # Create blank model
    nlp = spacy.blank("en")
    
    # Add NER component
    ner = nlp.add_pipe("ner")
    
    # Add labels
    for _, annotations in train_data:
        for ent in annotations.get("entities"):
            ner.add_label(ent[2])
    
    # Convert to Example objects
    examples = []
    for text, annotations in train_data:
        doc = nlp.make_doc(text)
        example = Example.from_dict(doc, annotations)
        examples.append(example)
    
    # Initialize the model
    nlp.initialize(lambda: examples)
    
    # Training loop
    print("🏋️ Training NER model...\n")
    
    for iteration in range(n_iter):
        random.shuffle(examples)
        losses = {}
        
        # Batch the examples
        batches = minibatch(examples, size=compounding(4.0, 32.0, 1.001))
        
        for batch in batches:
            nlp.update(batch, losses=losses)
        
        if (iteration + 1) % 10 == 0:
            print(f"   Iteration {iteration + 1}/{n_iter}, Loss: {losses['ner']:.4f}")
    
    print("\n✅ Training complete!")
    return nlp

# Train the model
custom_nlp = train_custom_ner(TRAIN_DATA, n_iter=30)

In [ ]:
# Test the custom model
test_sentences = [
    "Kotlin is a programming language developed by JetBrains.",
    "C++ was designed by Bjarne Stroustrup at Bell Labs.",
    "The Python programming language is very popular.",
    "Scala was created by Martin Odersky."
]

print("🧪 Testing Custom NER Model:\n")

for text in test_sentences:
    doc = custom_nlp(text)
    print(f"Text: {text}")
    if doc.ents:
        for ent in doc.ents:
            print(f"   • '{ent.text}' → {ent.label_}")
    else:
        print("   • No entities detected")
    print()

In [ ]:
# Save and load the model
import os

# Save
output_dir = "./custom_ner_model"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

custom_nlp.to_disk(output_dir)
print(f"✅ Model saved to {output_dir}")

# Load
loaded_nlp = spacy.load(output_dir)
print(f"✅ Model loaded from {output_dir}")

# Test loaded model
doc = loaded_nlp("Python is my favorite programming language.")
print(f"\nTest: {[(ent.text, ent.label_) for ent in doc.ents]}")

### Training with Pre-trained Models (Transfer Learning)

For better results, start with a pre-trained model:

In [ ]:
def train_on_pretrained(train_data, base_model="en_core_web_sm", n_iter=20):
    """Fine-tune a pre-trained model with new entity types"""
    
    # Load pre-trained model
    nlp = spacy.load(base_model)
    
    # Get the NER component
    ner = nlp.get_pipe("ner")
    
    # Add new labels
    for _, annotations in train_data:
        for ent in annotations.get("entities"):
            ner.add_label(ent[2])
    
    # Convert to Example objects
    examples = []
    for text, annotations in train_data:
        doc = nlp.make_doc(text)
        example = Example.from_dict(doc, annotations)
        examples.append(example)
    
    # Disable other pipes during training
    other_pipes = [pipe for pipe in nlp.pipe_names if pipe != "ner"]
    
    print(f"🏋️ Fine-tuning {base_model}...\n")
    
    with nlp.disable_pipes(*other_pipes):
        optimizer = nlp.resume_training()
        
        for iteration in range(n_iter):
            random.shuffle(examples)
            losses = {}
            
            batches = minibatch(examples, size=compounding(4.0, 32.0, 1.001))
            
            for batch in batches:
                nlp.update(batch, sgd=optimizer, losses=losses)
            
            if (iteration + 1) % 5 == 0:
                print(f"   Iteration {iteration + 1}/{n_iter}, Loss: {losses['ner']:.4f}")
    
    print("\n✅ Fine-tuning complete!")
    return nlp

# Fine-tune pre-trained model
finetuned_nlp = train_on_pretrained(TRAIN_DATA, n_iter=20)

In [ ]:
# Compare original vs fine-tuned
original_nlp = spacy.load("en_core_web_sm")

test_text = "Python was created by Guido van Rossum at Centrum Wiskunde & Informatica."

print("📊 Comparison: Original vs Fine-tuned\n")
print(f"Text: {test_text}\n")

print("Original model:")
doc_orig = original_nlp(test_text)
for ent in doc_orig.ents:
    print(f"   • '{ent.text}' → {ent.label_}")

print("\nFine-tuned model:")
doc_ft = finetuned_nlp(test_text)
for ent in doc_ft.ents:
    print(f"   • '{ent.text}' → {ent.label_}")

---

## 7. Best Practices & Tips

### Data Preparation

1. **Quality over Quantity**: Well-annotated data is more valuable than large amounts of noisy data
2. **Balanced Classes**: Ensure representation of all entity types
3. **Context Variation**: Include entities in different contexts
4. **Edge Cases**: Include tricky examples (overlapping entities, ambiguous cases)

### Training Tips

1. **Start with Pre-trained**: Always start with a pre-trained model when possible
2. **Learning Rate**: Use lower learning rates for fine-tuning (avoid catastrophic forgetting)
3. **Early Stopping**: Monitor validation loss to prevent overfitting
4. **Data Augmentation**: Create variations of your training data

### Production Deployment

1. **Model Size**: Choose appropriate model size for your latency requirements
2. **Batch Processing**: Use `nlp.pipe()` for processing multiple texts
3. **Disable Unused Components**: Only run necessary pipeline components
4. **GPU Acceleration**: Use spacy-transformers for GPU support

In [ ]:
# Best practices demonstration

# 1. Efficient batch processing
import time

texts = ["Sample text " + str(i) for i in range(100)]

# Bad: Processing one by one
start = time.time()
docs_slow = [nlp(text) for text in texts]
time_slow = time.time() - start

# Good: Using nlp.pipe()
start = time.time()
docs_fast = list(nlp.pipe(texts, batch_size=50))
time_fast = time.time() - start

print("⚡ Batch Processing Performance:")
print(f"   One-by-one: {time_slow:.3f}s")
print(f"   nlp.pipe(): {time_fast:.3f}s")
print(f"   Speedup: {time_slow/time_fast:.2f}x")

In [ ]:
# 2. Memory-efficient processing with n_process
print("\n💾 Memory-efficient processing:")

# For very large datasets, use n_process for multiprocessing
# Note: n_process > 1 requires if __name__ == "__main__" guard in scripts

# Process with generators to save memory
def process_large_dataset(texts, batch_size=100):
    """Process texts in batches using generators"""
    for doc in nlp.pipe(texts, batch_size=batch_size):
        # Yield only what you need
        yield {
            "text": doc.text[:50],
            "entities": [(ent.text, ent.label_) for ent in doc.ents]
        }

# Example usage
sample_texts = ["Apple CEO Tim Cook visited Berlin." for _ in range(5)]
results = list(process_large_dataset(sample_texts))
print(f"   Processed {len(results)} documents")
print(f"   Sample result: {results[0]}")

---

## 8. Further Reading

### 📚 Official Documentation

- [spaCy Documentation](https://spacy.io/usage)
- [EntityRecognizer API](https://spacy.io/api/entityrecognizer)
- [Rule-based Matching](https://spacy.io/usage/rule-based-matching)
- [Training Models](https://spacy.io/usage/training)

### 🔗 Tutorials

- [spaCy 101](https://spacy.io/usage/spacy-101)
- [NER with spaCy - Advanced NLP Course](https://course.spacy.io/)
- [Custom NER Training Guide](https://spacy.io/usage/training#ner)

### 📖 Papers

- [spaCy: Industrial-Strength NLP](https://spacy.io/)
- [A Primer on Neural Network Models for NLP](https://arxiv.org/abs/1510.00726)

---

## ✅ Lesson Summary

In this lesson, we covered:

1. **spaCy Basics**: Loading models, processing text, extracting entities
2. **Pipeline Architecture**: Understanding how NER fits in the pipeline
3. **Entity Rulers**: Rule-based NER for specific patterns
4. **Visualization**: Using displaCy for entity visualization
5. **Custom Training**: Training NER models from scratch and fine-tuning
6. **Best Practices**: Efficient processing and deployment tips

### 🚀 Next Lesson Preview

In **Lesson 3**, we'll dive into **BERT-based NER with Hugging Face Transformers**, including:
- Understanding BERT for token classification
- Using pre-trained NER models
- Fine-tuning BERT for custom NER tasks

In [ ]:
# Cleanup
import shutil
if os.path.exists("./custom_ner_model"):
    shutil.rmtree("./custom_ner_model")
    print("🧹 Cleaned up temporary model files")

print("\n🎉 Congratulations! You've completed Lesson 2: Traditional NER with spaCy")
print("\n👉 Continue to Lesson 3: BERT-based NER with Transformers")